In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
sys.path.append("..") 

from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel, FinalAnswerTool, GoogleSearchTool, VisitWebpageTool
import datetime
import requests
import pytz
import random
import yaml
import PIL
import numpy as np
from collections.abc import Iterable

import torch

from agents.utils import export_masks

from agents.tools.refiner.kitti_tracking import KittiDataset

from agents.Gradio_UI import GradioUI

from matplotlib import pyplot as plt

In [3]:
root_dir = "E:/KittiTracking"
n_steps, n_pred_steps = 16, 3

train_ds = KittiDataset(root_dir, "train", n_steps, n_pred_steps)
for sample in train_ds:
	frames, _ = sample
	break

In [11]:
web_search = GoogleSearchTool()
# search_tool = DuckDuckGoSearchTool(
#     max_results=5,
# )
visit_webpage = VisitWebpageTool()

In [ ]:
pages = search_tool(
    "What is Indoor Service Robot? What sensing modalities are necessary? What object classes are of interest?"
)
print(pages)

In [ ]:
urls = [
	"https://standardbots.com/blog/every-type-of-sensors-in-robotics---explained",
	"https://pmc.ncbi.nlm.nih.gov/articles/PMC10893033/",
	"https://www.sciencedirect.com/topics/computer-science/service-robot",
	"https://acroname.com/blog/sensors-robotics-5-common-types-0?srsltid=AfmBOor_I7R_vvfEdjwpbSzVXVJOFQcp5h5SgxinJwV9Wt-v0yA4a1nn"
	"https://ifr.org/img/office/Service_Robots_2016_Chapter_1_2.pdf"
]
for url in urls:
	print(url)
	whole_page = visit_webpage(url)
	
	print(whole_page)
	break

In [12]:
model = HfApiModel(
	max_tokens=4906,
	temperature=0.5,
	model_id="meta-llama/Meta-Llama-3-8B-Instruct", # it is possible that this model may be overloaded
	custom_role_conversions=None,
)

with open("./interpreter.yaml", 'r') as stream:
	prompt_templates = yaml.safe_load(stream)
# search_tool = DuckDuckGoSearchTool()
final_answer = FinalAnswerTool()

# final_answer 
agent = CodeAgent(
	model=model,
	tools=[
		web_search,
		visit_webpage,
		final_answer,
	],
	max_steps=6,
	verbosity_level=2,
    grammar=None,
	planning_interval=None,
	name=None,
	description=None,
	prompt_templates=prompt_templates,
)

result = agent.run(
	"Indoor Service Robot"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Indoor Service Robot                                                                                            │
│                                                                                                                 │
╰─ HfApiModel - meta-llama/Meta-Llama-3-8B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
---                                                                                                                
Task Understanding:                                                                                                
  Robots performing indoor services operate in indoor environments (homes, offices, hotels, campuses) to perform   
tasks such as cleaning, delivery, security, and assistance.                                                        
                                                                                                                   
Thought:                                                                                                           
  To accomplish this task, the robot needs to detect and navigate through indoor spaces.                           
  It requires sensing modalities to perceive its environment and extract relevant information.                     
  Let's identify the necessary sensing modalities and object classes of interest.                                  
                                                                                                                   
Required Modalities:                                                                                               
  1. RGB images: To detect objects, people, and obstacles using rich visual features.                              
  2. Depth images: To extract distances and spatial awareness of the environment.                                  
                                                                                                                   
Thought:                                                                                                           
  Now we know that the robot requires RGB and depth images to navigate and perform tasks.                          
  The next step is to identify the object classes of interest for the robot to detect and interact with.           
                                                                                                                   
Object Classes of Interest:                                                                                        
  1. Furnitures                                                                                                    
  2. Doors                                                                                                         
  3. Obstacles                                                                                                     
  4. Humans                                                                                                        
                                                                                                                   
Thought:                                                                                                           
  Now we have identified the necessary sensing modalities and object classes of interest for the robot.            
  The final step is to summarize the information and provide[38;2;230

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  print(("Robots performing indoor services operate in indoor environments (homes, offices, hotels, campuses) to   
  perform tasks such as cleaning, delivery, security, and assistance.", ['rgb', 'depth'], ["furnitures", "doors",  
  "obstacles", "humans"]))                                                                                         
  final_answer(("Robots performing indoor services operate in indoor environments (homes, offices, hotels,         
  campuses) to perform tasks such as cleaning, delivery, security, and assistance.", ['rgb', 'depth'],             
  ["furnitures", "doors", "obstacles", "humans"]))                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
('Robots performing indoor services operate in indoor environments (homes, offices, hotels, campuses) to perform 
tasks such as cleaning, delivery, security, and assistance.', ['rgb', 'depth'], ['furnitures', 'doors', 
'obstacles', 'humans'])

Out - Final answer: ('Robots performing indoor services operate in indoor environments (homes, offices, hotels, 
campuses) to perform tasks such as cleaning, delivery, security, and assistance.', ['rgb', 'depth'], ['furnitures',
'doors', 'obstacles', 'humans'])

[Step 1: Duration 5.09 seconds| Input tokens: 1,952 | Output tokens: 389]

In [14]:
description, modalities, obj_classes = result
result

('Robots performing indoor services operate in indoor environments (homes, offices, hotels, campuses) to perform tasks such as cleaning, delivery, security, and assistance.',
 ['rgb', 'depth'],
 ['furnitures', 'doors', 'obstacles', 'humans'])